This repo additionally requires the `pandas` library. Install via `pip install pandas`.

In [1]:
import pandas as pd
import pickle
import torch

In [2]:
def compute_pck_for_exp(results, anno_size,pck_bbox=True,per_img=False):

    alpha = torch.tensor([0.1, 0.05, 0.01])
    correct = torch.zeros(3)

    gt_correspondences, pred_correspondences, bbox_sizes, cats = [], [], [], []

    correct_all = []

    inds = []

    N_samples = len(results)
    for i_r in range(N_samples):
        r = results[i_r]
        vis = (r['src_kpts'][:,2] * r['trg_kpts'][:,2]) > 0
        single_gt_correspondences = torch.tensor(r['trg_kpts'][vis,:2])[:,[1,0]]
        single_pred_correspondences = torch.tensor(r['src_kpts_pred'][vis])[:,[1,0]]

        gt_correspondences.append(single_gt_correspondences)
        pred_correspondences.append(single_pred_correspondences)
        bbox_sizes.append(torch.tensor(r['threshold']).repeat(vis.sum()))
        

        if per_img:
            err = (single_gt_correspondences - single_pred_correspondences).norm(dim=-1)
            err = err.unsqueeze(0).repeat(3, 1)
            th = r['threshold']
            c_i = (err < alpha.unsqueeze(-1) * th).float().mean(dim=-1) if pck_bbox else (err < alpha.unsqueeze(-1) * anno_size).float().mean(dim=-1)
            cats += [r['category']] * 1
            correct += c_i / N_samples
            correct_all.append(c_i)
            inds.append(i_r)
        else:
            cats += [r['category']] * vis.sum()
            inds.append(torch.tensor(i_r).repeat(vis.sum()))

    if not per_img:
        inds = torch.cat(inds, dim=0).cpu()

    if not per_img:
        if type(gt_correspondences) == list:
            gt_correspondences = torch.cat(gt_correspondences, dim=0).cpu()
            pred_correspondences = torch.cat(pred_correspondences, dim=0).cpu()
            bbox_sizes = torch.cat(bbox_sizes, dim=0).cpu()

        alpha = torch.tensor([0.1, 0.05, 0.01])
        correct = torch.zeros(len(alpha))
        err = (pred_correspondences - gt_correspondences).norm(dim=-1)
        err = err.unsqueeze(0).repeat(len(alpha), 1)
        if pck_bbox:
            threshold = alpha.unsqueeze(-1) * bbox_sizes.unsqueeze(0)
            correct_all = err < threshold
        else:
            threshold = alpha * anno_size
            correct_all = err < threshold.unsqueeze(-1)

        correct_all = correct_all.permute(1,0)
        correct = correct_all.sum(dim=0) / len(gt_correspondences)
    else:
        correct_all = torch.stack(correct_all)
    
    d_df = {'category': cats}
    for i, a in enumerate(alpha):
        d_df[f'PCK_{a:0.2f}'] = correct_all[:,i].cpu().numpy()
    d_df['idx'] = inds

    df = pd.DataFrame(d_df)

    return df

In [6]:
def compute_and_plot_results(f_results, report_per_class=False):
    with open(f_results, 'rb') as file:
        results = pickle.load(file)

    anno_size = results['ANNO_SIZE']
    results = results['result']

    df = compute_pck_for_exp(results, anno_size,pck_bbox=True,per_img=True)
    print("[Per img] Micro :", df.iloc[:,1:-1].mean().round(4).values*100, "Macro :",df.groupby('category').mean().mean().round(4).values[:-1]*100)
    df = compute_pck_for_exp(results, anno_size,pck_bbox=True,per_img=False)
    if report_per_class:
        print('[Per kpt]',df.groupby('category').mean().iloc[:,0].round(4)*100)
    print("[Per kpt] Micro :", df.iloc[:,1:-1].mean().round(4).values*100, "Macro ",df.groupby('category').mean().mean().round(4).values[:-1]*100)

In [8]:

print("Spair")
f_results = f'../results/reported/results_0280_spair.pkl'
compute_and_plot_results(f_results)

print("IN3D")
f_results = f'../results/reported/results_0709_in3d.pkl'
compute_and_plot_results(f_results)

print("IN3D+Spair")
f_results = f'../results/reported/results_0294.pkl'
compute_and_plot_results(f_results)


print("DINO")
f_results = f'../results/reported/results_0300_dino_spair.pkl'
compute_and_plot_results(f_results)

print("DINO IN3D")
f_results = f'../results/reported/results_0712_dino_in3d.pkl'
compute_and_plot_results(f_results)

print("DINO IN3D+Spair")
f_results = f'../results/reported/results_0301_dino_in3d_spair.pkl'
compute_and_plot_results(f_results)


Spair
[Per img] Micro : [71.58     53.820004 10.14    ] Macro : [71.92 54.32 10.4 ]
[Per kpt] Micro : [74.43 56.76 11.22] Macro  [73.95 56.41 11.07]
IN3D
[Per img] Micro : [67.96 51.13  9.82] Macro : [68.47 51.72 10.08]
[Per kpt] Micro : [71.25 54.32 10.85] Macro  [70.82 54.01 10.7 ]
IN3D+Spair
[Per img] Micro : [72.24 54.64 10.55] Macro : [72.63 55.15 10.81]
[Per kpt] Micro : [75.06 57.51 11.52] Macro  [74.55 57.16 11.42]
DINO
[Per img] Micro : [71.25     51.35      9.059999] Macro : [71.8  51.98  9.32]
[Per kpt] Micro : [74.34 54.71 10.1 ] Macro  [73.61 54.06  9.92]
DINO IN3D
[Per img] Micro : [66.229996 47.82      8.45    ] Macro : [66.94 48.59  8.7 ]
[Per kpt] Micro : [69.55 51.07  9.25] Macro  [69.27 50.89  9.22]
DINO IN3D+Spair
[Per img] Micro : [69.92 50.21  8.61] Macro : [70.4  50.77  8.85]
[Per kpt] Micro : [73.36 53.63  9.62] Macro  [72.65 52.92  9.43]
